In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from sklearn.ensemble import RandomForestClassifier

# 1. PHYSICS ENGINE (Same as previous step)
def calculate_julian_date(dt):
    dt = dt.astimezone(timezone.utc)
    year, month, day = dt.year, dt.month, dt.day + dt.hour/24.0 + dt.minute/1440.0 + dt.second/86400.0
    if month <= 2: year -= 1; month += 12
    A = int(year / 100)
    B = 2 - A + int(A / 4)
    return int(365.25 * (year + 4716)) + int(30.6001 * (month + 1)) + day + B - 1524.5

def get_alignment_features(jd):
    T = (jd - 2451545.0) / 36525.0
    sun_lon = (280.466 + 36000.77 * T + 1.915 * np.sin(np.radians(357.529 + 35999.05 * T))) % 360
    M_moon_deg = (134.963 + 477198.8676 * T) % 360
    moon_lon = (218.316 + 481267.88 * T + 6.289 * np.sin(np.radians(M_moon_deg))) % 360
    moon_lat = 5.128 * np.sin(np.radians(93.27 + 483202.01 * T))
    sin_dec = np.sin(np.radians(moon_lat)) * np.cos(np.radians(23.439)) + \
              np.cos(np.radians(moon_lat)) * np.sin(np.radians(23.439)) * np.sin(np.radians(moon_lon))
    moon_dec = np.degrees(np.arcsin(sin_dec))
    
    def p_lon(L_m, pi, e): return (L_m + np.degrees(2 * e * np.sin(np.radians(L_m - pi)))) % 360
    sat_lon = p_lon(50.08 + 1222.1 * T, 92.06, 0.0555)
    jup_lon = p_lon(34.35 + 3034.9 * T, 14.33, 0.0485)
    mars_lon = p_lon(355.43 + 19140.3 * T, 336.06, 0.0934)
    
    moon_align = (1 + np.cos(np.radians(2 * (moon_lon - sun_lon)))) / 2
    dec_abs = abs(moon_dec)
    sat_align = (1 + np.cos(np.radians(2 * (sat_lon - sun_lon)))) / 2
    jup_align = (1 + np.cos(np.radians(2 * (jup_lon - sun_lon)))) / 2
    mars_align = (1 + np.cos(np.radians(2 * (mars_lon - sun_lon)))) / 2
    lunar_dist = (1 + np.cos(np.radians(M_moon_deg))) / 2
    return [moon_align, dec_abs, sat_align, jup_align, mars_align, lunar_dist]

# --- THE GAME CHANGER: 7-DAY ROLLING MAXIMUM ---
def get_peak_features_7_days(target_dt):
    """Scans the 7 days prior to the target date and returns the maximum stress values."""
    daily_features = []
    # Check every 12 hours for the 7 days leading up to the quake
    for hours_back in range(0, 7 * 24, 12):
        check_dt = target_dt - timedelta(hours=hours_back)
        jd = calculate_julian_date(check_dt)
        daily_features.append(get_alignment_features(jd))
    
    # Return the maximum peak alignment found in that 7-day window
    return np.max(daily_features, axis=0).tolist()

# ==========================================
# 2. LOAD DATA USING THE 7-DAY WINDOW
# ==========================================
print("⏳ Loading Quakes (Finding Peak Stress in 7-Days Prior)...")
df_real = pd.read_csv('/workspaces/AI-Catastrophe-Analytics/notebooks/megaquake/Inputs_Mega_Quake_Topocentric_Analysis.csv')
df_real['time'] = pd.to_datetime(df_real['time'], utc=True, format='mixed', errors='coerce')


X_real = []
for _, row in df_real.dropna(subset=['time']).iterrows():
    X_real.append(get_peak_features_7_days(row['time']))
y_real = [1] * len(X_real)

print("⏳ Generating Synthetic Background...")
np.random.seed(42)
start_dt = datetime(1900, 1, 1, tzinfo=timezone.utc)
total_seconds = int((datetime(2025, 12, 31, tzinfo=timezone.utc) - start_dt).total_seconds())

X_syn = []
# Sample directly from bounds to avoid materializing a huge range
for sec in np.random.randint(0, total_seconds, size=10000):
    dt = start_dt + timedelta(seconds=int(sec))
    X_syn.append(get_alignment_features(calculate_julian_date(dt)))
y_syn = [0] * len(X_syn)

X = np.array(X_real + X_syn)
y = np.array(y_real + y_syn)

# ==========================================
# 3. TRAINING AND TESTING
# ==========================================
print("🧠 Training AI with Stress Loading Physics...")
rf_model = RandomForestClassifier(n_estimators=500, class_weight='balanced', max_depth=12, random_state=42)
rf_model.fit(X, y)

historic_events = [
    {"name": "2001 Bhuj (Gujarat)", "time": "2001-01-26 03:14:40"},
    {"name": "2004 Sumatra-Andaman", "time": "2004-12-26 00:58:53"},
    {"name": "2011 Tohoku (Japan)", "time": "2011-03-11 05:46:24"}
]

print("\n🚨 TESTING AI MODEL (WITH 7-DAY LOADING WINDOW):")
for ev in historic_events:
    dt = datetime.strptime(ev['time'], "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    
    # Predict based on the peak stress in the 7 days prior to the event
    feats = np.array([get_peak_features_7_days(dt)])
    risk_probability = rf_model.predict_proba(feats)[0][1] * 100 
    
    alert_level = "🔴 EXTREME" if risk_probability > 85 else "🟡 HIGH" if risk_probability > 60 else "⚪ NORMAL"
    print(f"{ev['name']}: -> Peak Stress AI Probability: {risk_probability:.1f}% [{alert_level}]")
    
    # Dynamic Alert Level
    if risk_probability > 85:
        alert_level = "🔴 EXTREME"
    elif risk_probability > 65:
        alert_level = "🟡 HIGH"
    elif risk_probability > 50:
        alert_level = "🔵 ELEVATED"
    else:
        alert_level = "⚪ NORMAL"
        
    print(f"{ev['name']}:")
    print(f"   -> AI Risk Probability: {risk_probability:.1f}% [{alert_level}]")

⏳ Loading Quakes (Finding Peak Stress in 7-Days Prior)...
⏳ Generating Synthetic Background...
🧠 Training AI with Stress Loading Physics...

🚨 TESTING AI MODEL (WITH 7-DAY LOADING WINDOW):
2001 Bhuj (Gujarat): -> Peak Stress AI Probability: 80.1% [🟡 HIGH]
2001 Bhuj (Gujarat):
   -> AI Risk Probability: 80.1% [🟡 HIGH]
2004 Sumatra-Andaman: -> Peak Stress AI Probability: 74.5% [🟡 HIGH]
2004 Sumatra-Andaman:
   -> AI Risk Probability: 74.5% [🟡 HIGH]
2011 Tohoku (Japan): -> Peak Stress AI Probability: 86.0% [🔴 EXTREME]
2011 Tohoku (Japan):
   -> AI Risk Probability: 86.0% [🔴 EXTREME]


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. THE PHYSICS ENGINE
# ==========================================
def calculate_julian_date(dt):
    dt = dt.astimezone(timezone.utc)
    year, month, day = dt.year, dt.month, dt.day + dt.hour/24.0 + dt.minute/1440.0 + dt.second/86400.0
    if month <= 2: year -= 1; month += 12
    A = int(year / 100)
    B = 2 - A + int(A / 4)
    return int(365.25 * (year + 4716)) + int(30.6001 * (month + 1)) + day + B - 1524.5

def get_alignment_features(jd):
    T = (jd - 2451545.0) / 36525.0
    sun_lon = (280.466 + 36000.77 * T + 1.915 * np.sin(np.radians(357.529 + 35999.05 * T))) % 360
    M_moon_deg = (134.963 + 477198.8676 * T) % 360
    moon_lon = (218.316 + 481267.88 * T + 6.289 * np.sin(np.radians(M_moon_deg))) % 360
    moon_lat = 5.128 * np.sin(np.radians(93.27 + 483202.01 * T))
    
    sin_dec = np.sin(np.radians(moon_lat)) * np.cos(np.radians(23.439)) + \
              np.cos(np.radians(moon_lat)) * np.sin(np.radians(23.439)) * np.sin(np.radians(moon_lon))
    moon_dec = np.degrees(np.arcsin(sin_dec))
    
    def p_lon(L_m, pi, e): return (L_m + np.degrees(2 * e * np.sin(np.radians(L_m - pi)))) % 360
    sat_lon = p_lon(50.08 + 1222.1 * T, 92.06, 0.0555)
    jup_lon = p_lon(34.35 + 3034.9 * T, 14.33, 0.0485)
    mars_lon = p_lon(355.43 + 19140.3 * T, 336.06, 0.0934)
    
    moon_align = (1 + np.cos(np.radians(2 * (moon_lon - sun_lon)))) / 2
    dec_abs = abs(moon_dec)
    sat_align = (1 + np.cos(np.radians(2 * (sat_lon - sun_lon)))) / 2
    jup_align = (1 + np.cos(np.radians(2 * (jup_lon - sun_lon)))) / 2
    mars_align = (1 + np.cos(np.radians(2 * (mars_lon - sun_lon)))) / 2
    lunar_dist = (1 + np.cos(np.radians(M_moon_deg))) / 2
    
    return [moon_align, dec_abs, sat_align, jup_align, mars_align, lunar_dist]

def get_peak_features_7_days(target_dt):
    """Scans the 7 days prior to the target date and returns peak stress values."""
    daily_features = []
    for hours_back in range(0, 7 * 24, 12):
        jd = calculate_julian_date(target_dt - timedelta(hours=hours_back))
        daily_features.append(get_alignment_features(jd))
    return np.max(daily_features, axis=0).tolist()

# ==========================================
# 2. TRAIN THE AI MODEL
# ==========================================
print("⏳ Phase 1: Training the AI on Historical 'Black Swans'...")
try:
    df_real = pd.read_csv('/workspaces/AI-Catastrophe-Analytics/notebooks/megaquake/Inputs_Mega_Quake_Topocentric_Analysis.csv')
    df_real['time'] = pd.to_datetime(df_real['time'], utc=True, format='mixed', errors='coerce')

except FileNotFoundError:
    print("Error: 'Mega_Quake_Formula_Inputs.csv' not found. Please ensure it is in the same directory.")
    exit()

X_real = [get_peak_features_7_days(row['time']) for _, row in df_real.dropna(subset=['time']).iterrows()]
y_real = [1] * len(X_real)

np.random.seed(42)
start_dt = datetime(1900, 1, 1, tzinfo=timezone.utc)
total_seconds = int((datetime(2025, 12, 31, tzinfo=timezone.utc) - start_dt).total_seconds())
X_syn = [get_alignment_features(calculate_julian_date(start_dt + timedelta(seconds=int(sec))))
         for sec in np.random.randint(0, total_seconds, size=10000)]
y_syn = [0] * len(X_syn)

rf_model = RandomForestClassifier(n_estimators=500, class_weight='balanced', max_depth=12, random_state=42)
rf_model.fit(np.array(X_real + X_syn), np.array(y_real + y_syn))
print("✅ AI Training Complete. Accuracy matrix heavily penalizes false negatives.")

# ==========================================
# 3. THE 60-MONTH FUTURE SWEEP (2026 - 2031)
# ==========================================
print("\n🚀 Phase 2: Scanning Future Dates (2026-2031)...")

try:
    mask = pd.read_csv('/workspaces/AI-Catastrophe-Analytics/notebooks/megaquake/subduction_zones.csv')
    fault_zones = mask['zone'].unique()
except FileNotFoundError:
    print("Warning: 'subduction_zones.csv' not found. Will output dates without geographic mapping.")
    fault_zones = ["Global Watch"]

forecast_start = datetime(2026, 1, 1, tzinfo=timezone.utc)
forecast_end = forecast_start + timedelta(days=365 * 5)

current_dt = forecast_start
high_risk_windows = []

# Sweep the future by 1-day increments
while current_dt <= forecast_end:
    # Get peak stress of the preceding 7 days
    feats = np.array([get_peak_features_7_days(current_dt)])
    risk_prob = rf_model.predict_proba(feats)[0][1] * 100
    
    # We only care about Extreme AI predictions (> 75%)
    if risk_prob > 75.0:
        for zone in fault_zones:
            high_risk_windows.append({
                'Risk_Date': current_dt.strftime('%Y-%m-%d'),
                'Peak_Stress_Probability': round(risk_prob, 1),
                'Alert_Level': '🔴 EXTREME' if risk_prob > 85 else '🟡 HIGH',
                'Threatened_Zone': zone
            })
            
    current_dt += timedelta(days=1)
    
    if current_dt.month == 1 and current_dt.day == 1:
        print(f"   -> Scanned through {current_dt.year}...")

# ==========================================
# 4. EXPORT FORECAST
# ==========================================
if high_risk_windows:
    df_forecast = pd.DataFrame(high_risk_windows)
    
    # Remove consecutive duplicate alerts for the same zone to clean up the CSV
    df_forecast = df_forecast.drop_duplicates(subset=['Risk_Date', 'Threatened_Zone'])
    
    output_filename = 'AI_Mega_Quake_Forecast_2026_2031.csv'
    df_forecast.to_csv(output_filename, index=False)
    
    print(f"\n✅ Forecast Complete! Found {len(df_forecast['Risk_Date'].unique())} high-risk dates.")
    print(f"💾 Results saved to '{output_filename}'.")
else:
    print("\n✅ Forecast Complete. The AI did not detect any >75% Extreme Risk patterns in the next 5 years.")

⏳ Phase 1: Training the AI on Historical 'Black Swans'...


MemoryError: 